In [3]:
pip install behave sentence-transformers pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 223.6/223.6 kB 6.0 MB/s eta 0:00:00


In [4]:
"""
FULL DATASET ANALYSIS
Mở rộng từ pilot_analysis.ipynb để chạy trên toàn bộ dataset (full_llm_output.csv).

Khác biệt so với pilot:
- Pilot dùng `mock_ground_truth`: dict viết tay 10 kịch bản Gherkin mẫu.
- Ở đây KHÔNG có Gherkin ground truth viết tay cho 50 story, nên ta build
  một "reference text" cho mỗi story từ (role, goal, benefit, expected_test_scenarios)
  trong full_ground_truth.csv, theo format Connextra + acceptance criteria.
  Vì SentenceTransformer đo similarity ngữ nghĩa (không quan tâm format
  Gherkin hay văn xuôi), cách này giữ đúng tinh thần đo lường của RQ1 trong pilot.
- full_llm_output.csv thực ra đã merge sẵn toàn bộ cột của full_ground_truth.csv
  (đã kiểm tra: các cột trùng tên có giá trị giống hệt nhau), nên script này
  chỉ cần đọc 1 file duy nhất. Nếu bạn muốn tách biệt rõ ràng, có thể merge
  thủ công bằng story_id (code merge cũng được để sẵn bên dưới, đã comment).

Requirements: pip install behave sentence-transformers pandas
"""

import pandas as pd
import re
import numpy as np
from sentence_transformers import SentenceTransformer, util
from behave.parser import parse_feature, ParserError

# ==========================================
# PHẦN 1: KHỞI TẠO
# ==========================================
print("1. Đang tải model NLP (SentenceTransformers)...")
model = SentenceTransformer('all-MiniLM-L6-v2')

LLM_OUTPUT_CSV = 'full_llm_output.csv'
GROUND_TRUTH_CSV = 'full_ground_truth.csv'   # dùng nếu bạn muốn merge tách biệt
OUTPUT_CSV = 'full_measurement_results.csv'


# ==========================================
# PHẦN 2: CÁC HÀM XỬ LÝ LÕI (giữ nguyên logic từ pilot)
# ==========================================
def extract_gherkin(text):
    """Trích xuất mã Gherkin từ output của LLM."""
    if pd.isna(text):
        return ""
    text_str = str(text)

    match = re.search(r'```(?:gherkin|feature)?\n(.*?)\n```', text_str, re.IGNORECASE | re.DOTALL)
    if match:
        extracted = match.group(1).strip()
        if 'Feature:' in extracted:
            return extracted

    if 'Feature:' in text_str:
        lines = text_str.split('\n')
        gherkin_lines = []
        is_gherkin = False
        for line in lines:
            if line.strip().startswith('Feature:'):
                is_gherkin = True
            if is_gherkin:
                if line.strip().startswith('```') or line.strip().startswith('#'):
                    break
                gherkin_lines.append(line)
        return '\n'.join(gherkin_lines).strip()
    return ""


def check_syntax_validity(gherkin_text):
    """
    Kiểm tra độ hợp lệ cú pháp (Syntax) bằng Behave Parser thuần túy (RQ2).
    Trả về (score 0/1, message).
    """
    if not gherkin_text or 'Feature:' not in gherkin_text:
        return 0, "Lỗi: Code rỗng hoặc thiếu từ khóa 'Feature:'"

    try:
        parse_feature(gherkin_text)
        return 1, "Hợp lệ (Cú pháp chuẩn xác)"
    except ParserError as e:
        error_line = str(e).strip().split('\n')[0]
        return 0, f"Sai cú pháp Gherkin: {error_line}"
    except Exception as e:
        return 0, f"Lỗi khác: {str(e)}"


def build_reference_text(row):
    """
    Dựng 'ground truth' dạng văn bản cho RQ1 từ role/goal/benefit +
    expected_test_scenarios, thay cho Gherkin viết tay trong pilot.
    """
    role = str(row.get('role', '')).strip()
    goal = str(row.get('goal', '')).strip()
    benefit = str(row.get('benefit', '')).strip()
    criteria = str(row.get('expected_test_scenarios', '')).strip()

    story = f"As a {role}, I want {goal}, so that {benefit}."
    if criteria and criteria.lower() != 'nan':
        story += f" Acceptance criteria: {criteria}"
    return story


# ==========================================
# PHẦN 3: PIPELINE CHÍNH
# ==========================================
def run_full_pipeline(llm_csv_path, ground_truth_csv_path=None):
    print(f"2. Đang đọc dữ liệu từ file {llm_csv_path}...")
    try:
        df = pd.read_csv(llm_csv_path)
    except FileNotFoundError:
        print(f"LỖI: Không tìm thấy file {llm_csv_path}.")
        return None

    # Nếu file llm_output CHƯA có sẵn các cột role/goal/benefit/expected_test_scenarios,
    # merge thêm từ ground truth. (full_llm_output.csv hiện đã có sẵn nên bước này
    # thường là no-op / bị bỏ qua an toàn.)
    required_gt_cols = {'role', 'goal', 'benefit', 'expected_test_scenarios'}
    if ground_truth_csv_path and not required_gt_cols.issubset(df.columns):
        gt = pd.read_csv(ground_truth_csv_path)
        df = df.merge(gt[['story_id'] + list(required_gt_cols)], on='story_id', how='left')

    # Trích xuất Gherkin
    print("3. Đang trích xuất mã Gherkin từ nội dung LLM...")
    df['extracted_gherkin'] = df['llm_output'].apply(extract_gherkin)

    # RQ2: Syntax validity
    print("4. Đang kiểm tra cú pháp (RQ2)...")
    rq2_results = df['extracted_gherkin'].apply(check_syntax_validity)
    df['rq2_validity'] = [res[0] for res in rq2_results]
    df['rq2_message'] = [res[1] for res in rq2_results]

    # RQ1: Semantic similarity
    print("5. Đang đo lường độ tương đồng ngữ nghĩa (RQ1)...")
    df['reference_text'] = df.apply(build_reference_text, axis=1)

    semantic_scores = []
    for _, row in df.iterrows():
        llm_gherkin = row['extracted_gherkin']
        ground_truth = row['reference_text']

        if llm_gherkin and ground_truth:
            emb1 = model.encode(llm_gherkin, convert_to_tensor=True)
            emb2 = model.encode(ground_truth, convert_to_tensor=True)
            semantic_scores.append(util.cos_sim(emb1, emb2).item())
        else:
            semantic_scores.append(np.nan)

    df['rq1_similarity'] = semantic_scores

    return df


# ==========================================
# KÍCH HOẠT CHẠY
# ==========================================
if __name__ == "__main__":
    final_df = run_full_pipeline(LLM_OUTPUT_CSV, GROUND_TRUTH_CSV)

    if final_df is not None:
        print("\n" + "=" * 40)
        print("HOÀN THÀNH! TÓM TẮT KẾT QUẢ:")
        print("=" * 40)

        print(final_df[['story_id', 'rq2_validity', 'rq1_similarity']].to_string(index=False))

        print("\n--- Thống kê tổng quan ---")
        print(f"RQ1 - Similarity trung bình: {final_df['rq1_similarity'].mean():.4f}")
        print(f"RQ1 - Similarity độ lệch chuẩn: {final_df['rq1_similarity'].std():.4f}")
        print(f"RQ2 - Tỉ lệ hợp lệ cú pháp: {final_df['rq2_validity'].mean() * 100:.2f}%  "
              f"({final_df['rq2_validity'].sum()}/{len(final_df)})")

        final_df.to_csv(OUTPUT_CSV, index=False)
        print(f"\nĐã xuất file kết quả: {OUTPUT_CSV}")


1. Đang tải model NLP (SentenceTransformers)...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

2. Đang đọc dữ liệu từ file full_llm_output.csv...
3. Đang trích xuất mã Gherkin từ nội dung LLM...
4. Đang kiểm tra cú pháp (RQ2)...
5. Đang đo lường độ tương đồng ngữ nghĩa (RQ1)...

HOÀN THÀNH! TÓM TẮT KẾT QUẢ:
story_id  rq2_validity  rq1_similarity
   US001             1        0.730303
   US002             1        0.690617
   US003             1        0.674130
   US004             1        0.865011
   US005             1        0.676602
   US006             1        0.817156
   US007             1        0.836635
   US008             1        0.670045
   US009             1        0.780648
   US010             1        0.792083
   US011             1        0.759138
   US012             1        0.698935
   US013             1        0.702190
   US014             1        0.767478
   US015             1        0.898921
   US016             1        0.751022
   US017             1        0.749016
   US018             1        0.871719
   US019             1        0.726759
   US0

In [5]:
"""
RQ1 — INTER-RATER RELIABILITY (Cohen's Kappa) TRÊN 20% SAMPLE NGẪU NHIÊN
=========================================================================
Mục tiêu: RQ1 dùng SentenceTransformer để đo similarity ngữ nghĩa (điểm liên
tục 0-1) giữa Gherkin do LLM sinh ra và ground truth. Để chứng minh độ tin
cậy (effect size / reliability) của cách đo này, ta cần 2 người đánh giá độc
lập (annotator) gán nhãn PHÂN LOẠI (categorical) cho cùng một mẫu ngẫu nhiên
= 20% số story, rồi tính Cohen's kappa giữa 2 người. Mục tiêu: κ ≥ 0.70
(mức "Substantial agreement" trở lên theo thang Landis & Koch).

QUY TRÌNH SỬ DỤNG:
  BƯỚC 1: Chạy hàm `export_irr_sample()` -> xuất ra file CSV mẫu 20%,
           có sẵn 2 cột trống 'annotator_1_label' và 'annotator_2_label'.
  BƯỚC 2: Gửi file này cho 2 annotator, để họ ĐỘC LẬP đọc extracted_gherkin
           và reference_text rồi điền nhãn theo rubric bên dưới (KHÔNG được
           thấy nhãn của nhau).
  BƯỚC 3: Chạy hàm `compute_kappa()` trên file đã điền đầy đủ.

RUBRIC GÁN NHÃN GỢI Ý (có thể điều chỉnh theo đề tài, miễn 2 annotator dùng
chung 1 rubric):
  2 = Match       : Gherkin của LLM thể hiện đúng, đầy đủ ý nghĩa của ground truth
  1 = Partial      : Có liên quan nhưng thiếu/sai một phần ý nghĩa quan trọng
  0 = No match     : Không phản ánh đúng ý nghĩa của ground truth

Requirements: pip install pandas scikit-learn
"""

import pandas as pd
import numpy as np
from sklearn.metrics import cohen_kappa_score

RESULTS_CSV = 'full_measurement_results.csv'   # output của full_analysis.py
IRR_SAMPLE_CSV = 'rq1_irr_sample_TO_ANNOTATE.csv'
RANDOM_SEED = 42
SAMPLE_FRACTION = 0.20
KAPPA_THRESHOLD = 0.70


# ==========================================
# BƯỚC 1: XUẤT MẪU 20% ĐỂ 2 ANNOTATOR GÁN NHÃN
# ==========================================
def export_irr_sample(results_csv=RESULTS_CSV,
                       out_csv=IRR_SAMPLE_CSV,
                       frac=SAMPLE_FRACTION,
                       seed=RANDOM_SEED):
    df = pd.read_csv(results_csv)

    n_total = len(df)
    sample = df.sample(frac=frac, random_state=seed).sort_values('story_id')
    n_sample = len(sample)

    cols_to_keep = ['story_id', 'extracted_gherkin', 'reference_text', 'rq1_similarity']
    cols_to_keep = [c for c in cols_to_keep if c in sample.columns]

    out = sample[cols_to_keep].copy()
    out['annotator_1_label'] = ""   # để trống cho annotator 1 điền (0/1/2)
    out['annotator_2_label'] = ""   # để trống cho annotator 2 điền (0/1/2)

    out.to_csv(out_csv, index=False)
    print(f"Đã lấy mẫu ngẫu nhiên {n_sample}/{n_total} story (seed={seed}).")
    print(f"Đã xuất file để annotator gán nhãn: {out_csv}")
    print("Nhắc: 2 annotator điền cột 'annotator_1_label' / 'annotator_2_label' "
          "ĐỘC LẬP với nhau (0=No match, 1=Partial, 2=Match).")
    return out


# ==========================================
# BƯỚC 2: TÍNH COHEN'S KAPPA (effect size / độ tin cậy liên đánh giá)
# ==========================================
def compute_kappa(annotated_csv=IRR_SAMPLE_CSV,
                   col1='annotator_1_label',
                   col2='annotator_2_label',
                   weights=None,
                   threshold=KAPPA_THRESHOLD):
    """
    weights: None -> unweighted kappa (dùng khi nhãn là danh mục, không có thứ tự
                      quan trọng giữa các mức)
             'linear' hoặc 'quadratic' -> weighted kappa (khuyến nghị nếu nhãn
                      0/1/2 có TÍNH THỨ TỰ, vì sai lệch 0 vs 2 nghiêm trọng hơn
                      0 vs 1). Với rubric 3 mức Match/Partial/No-match, nên
                      dùng weights='linear' hoặc 'quadratic'.
    """
    df = pd.read_csv(annotated_csv)

    missing = df[col1].isna().sum() + df[col2].isna().sum()
    if missing > 0:
        raise ValueError(
            f"Còn {missing} ô nhãn trống trong '{annotated_csv}'. "
            "Hãy đảm bảo cả 2 annotator đã điền đầy đủ trước khi tính kappa."
        )

    labels1 = df[col1].astype(int)
    labels2 = df[col2].astype(int)

    kappa = cohen_kappa_score(labels1, labels2, weights=weights)

    # Thang diễn giải Landis & Koch (1977)
    def interpret(k):
        if k < 0:
            return "Poor (kém hơn ngẫu nhiên)"
        elif k < 0.20:
            return "Slight"
        elif k < 0.40:
            return "Fair"
        elif k < 0.60:
            return "Moderate"
        elif k < 0.80:
            return "Substantial"
        else:
            return "Almost Perfect"

    weight_label = weights if weights else "unweighted"
    print("=" * 50)
    print(f"COHEN'S KAPPA (RQ1 Inter-Rater Reliability, n={len(df)}, weights={weight_label})")
    print("=" * 50)
    print(f"κ = {kappa:.4f}  ->  {interpret(kappa)}")

    if kappa >= threshold:
        print(f"✅ ĐẠT ngưỡng yêu cầu κ ≥ {threshold:.2f}")
    else:
        print(f"❌ CHƯA ĐẠT ngưỡng yêu cầu κ ≥ {threshold:.2f}. "
              "Cân nhắc: làm rõ lại rubric, thảo luận và thống nhất giữa "
              "2 annotator, hoặc annotate lại các case bất đồng.")

    # In ra các case 2 annotator bất đồng, hữu ích để thảo luận/thống nhất
    disagreements = df[labels1 != labels2]
    if len(disagreements) > 0:
        print(f"\n{len(disagreements)} story có bất đồng giữa 2 annotator:")
        print(disagreements[['story_id', col1, col2]].to_string(index=False))

    return kappa


# ==========================================
# CHẠY THỬ / VÍ DỤ
# ==========================================
if __name__ == "__main__":
    # BƯỚC 1 — chạy trước, sau đó gửi file cho 2 annotator điền nhãn:
    export_irr_sample()

    # BƯỚC 2 — chạy SAU KHI 2 annotator đã điền xong file rq1_irr_sample_TO_ANNOTATE.csv:
    # compute_kappa(weights='linear')


Đã lấy mẫu ngẫu nhiên 10/50 story (seed=42).
Đã xuất file để annotator gán nhãn: rq1_irr_sample_TO_ANNOTATE.csv
Nhắc: 2 annotator điền cột 'annotator_1_label' / 'annotator_2_label' ĐỘC LẬP với nhau (0=No match, 1=Partial, 2=Match).


In [12]:
compute_kappa(annotated_csv='rq1_irr_sample_TO_ANNOTATE.csv', weights='linear')

COHEN'S KAPPA (RQ1 Inter-Rater Reliability, n=10, weights=linear)
κ = 0.7619  ->  Substantial
✅ ĐẠT ngưỡng yêu cầu κ ≥ 0.70

1 story có bất đồng giữa 2 annotator:
story_id  annotator_1_label  annotator_2_label
   US014                  2                  1


np.float64(0.7619047619047619)